In [1]:

%%capture
!pip install tensorboard
!pip install nltk
!pip install transformers
!pip install datasets
!pip install sacrebleu accelerate
!pip install git+https://github.com/csebuetnlp/normalizer

In [3]:
import shutil
import os

folder_path = "/kaggle/working/output_dir"  # replace with your folder

# Check if folder exists
if os.path.exists(folder_path):
    shutil.rmtree(folder_path)   # deletes folder and all contents
    print(f"Deleted {folder_path}")
else:
    print("Folder does not exist")

zip_path = "/kaggle/working/model_checkpoints.zip"  # path to your zip file

# Check if file exists
if os.path.exists(zip_path):
    os.remove(zip_path)   # deletes the zip file
    print(f"Deleted {zip_path}")
else:
    print("File does not exist")


Folder does not exist
File does not exist


In [ ]:
!pip install evaluate

In [4]:
%%capture
import torch
import numpy as np
import pandas as pd
from datasets import Dataset
from normalizer import normalize
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer,
    EarlyStoppingCallback
)
import nltk
from transformers.integrations import TensorBoardCallback
from transformers import TrainerCallback
# Ensure the tokenizer data is downloaded (required in Kaggle/Colab)
nltk.download('punkt')
nltk.download('punkt_tab') # Required for newer NLTK versions
# Verify installation
import transformers
print(f"Transformers version: {transformers.__version__}")

2026-02-08 06:39:00.652852: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770532740.877275      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770532740.939287      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770532741.481987      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770532741.482034      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770532741.482037      55 computation_placer.cc:177] computation placer alr

In [5]:
tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M", src_lang="ben_Beng" , tgt_lang="eng_Latn")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M")

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [ ]:
for name, param in model.model.encoder.named_parameters():
    if param.requires_grad:
        print(f"{name} is trainable")
    else:
        print(f"{name} is frozen")

In [6]:
# Freeze all encoder layers initially
for param in model.model.encoder.parameters():
    param.requires_grad = False

print("All encoder layers frozen initially.")

All encoder layers frozen initially.


In [7]:
from transformers import TrainerCallback


class GradualUnfreezingCallback(TrainerCallback):
    """
    Gradually unfreeze encoder layers during training.

    unfreeze_schedule: dict mapping epoch -> number of encoder layers to unfreeze
    Example:
        {
            1: 2,
            2: 4,
            3: "all"
        }
    """

    def __init__(self, model, unfreeze_schedule):
        self.model = model
        self.unfreeze_schedule = unfreeze_schedule

        # Encoder layers (works for most encoder-decoder models)
        self.encoder_layers = model.model.encoder.layers
        self.total_layers = len(self.encoder_layers)

    def on_epoch_begin(self, args, state, control, **kwargs):
        trainer = kwargs.get("trainer", None)
        if trainer is None:
            return

        # Trainer epochs are floats (e.g., 1.0, 2.0)
        epoch = int(state.epoch)

        if epoch not in self.unfreeze_schedule:
            return

        layers_to_unfreeze = self.unfreeze_schedule[epoch]

        if layers_to_unfreeze == "all":
            start = 0
        else:
            start = max(0, self.total_layers - layers_to_unfreeze)

        print(
            f"\n🔓 Unfreezing encoder layers "
            f"{start} to {self.total_layers - 1} at epoch {epoch}"
        )

        # Unfreeze selected layers
        for layer in self.encoder_layers[start:]:
            for param in layer.parameters():
                param.requires_grad = True

        # IMPORTANT: rebuild optimizer so new params are included
        trainer.create_optimizer()

        return control


In [8]:
tensorboard_callback = TensorBoardCallback()

In [9]:
max_length = 128

In [10]:
def tokenize_and_create_dataset(tokenizer, data_df, max_length=128, normalize_source=True):
    """
    Tokenizes source (Bangla) and target (English) texts for MT fine-tuning.
    Applies optional normalization on Bangla input.
    """
    # Normalize Bangla if requested
    if normalize_source:
        source_texts = [normalize(text) for text in data_df["bangla_speech"]]
    else:
        source_texts = list(data_df["bangla_speech"])

    target_texts = list(data_df["english_speech"])
    print("\n=== Checking types of source and target texts ===")
    for i in range(min(5, len(source_texts))):
        print(f"Example {i+1}: Source type={type(source_texts[i])}, Target type={type(target_texts[i])}")
    # Tokenize source
    encodings = tokenizer(
        source_texts,
        truncation=True,
        max_length=max_length
    )

    # Tokenize target with text_target=True
    decodings = tokenizer(
        target_texts,
        truncation=True,
        max_length=max_length
       # text_target=True
    )

    # Replace padding tokens with -100
    labels = [
        [-100 if token == tokenizer.pad_token_id else token for token in l]
        for l in decodings["input_ids"]
    ]

    # DEBUG: Decode first 3 examples
    print("\n=== First 3 tokenized examples ===")
    for i in range(min(3, len(encodings["input_ids"]))):
        # Decode input_ids
        decoded_input = tokenizer.decode(encodings["input_ids"][i], skip_special_tokens=True)
        # Decode labels (replace -100 back to pad_token_id)
        label_ids_for_decoding = [token if token != -100 else tokenizer.pad_token_id for token in labels[i]]
        decoded_label = tokenizer.decode(label_ids_for_decoding, skip_special_tokens=True)
        print(f"\nExample {i+1}:")
        print(f"Input : {decoded_input}")
        print(f"Label : {decoded_label}")

    # Build Dataset
    dataset = Dataset.from_dict({
        "input_ids": encodings["input_ids"],
        "attention_mask": encodings["attention_mask"],
        "labels": labels
    })

    return dataset


data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    padding="max_length",   # ensure consistent length
    max_length=128,          # optional but safer
    label_pad_token_id=-100 # recommended for Seq2Seq to ignore pad in loss
)

In [11]:
# Path to your CSV file
csv_path = "/kaggle/input/train-fin-fin/all_train_dataset_final.csv"

# Load CSV into a DataFrame
train_df = pd.read_csv(csv_path)

# Fill missing values with empty strings to avoid tokenizer errors
train_df["bangla_speech"] = train_df["bangla_speech"].fillna("")
train_df["english_speech"] = train_df["english_speech"].fillna("")

# Ensure all entries are strings
train_df["bangla_speech"] = train_df["bangla_speech"].astype(str)
train_df["english_speech"] = train_df["english_speech"].astype(str)

# Quick check
print(train_df.head())
print(f"DataFrame shape: {train_df.shape}")



                                       bangla_speech  \
0                              ঢাকায় আমার চাকরি হইসে   
1                     আবার চল্যিত শুরু গরিল বেজ্ঞুনে   
2  কিতা কইতাম ... বাহবা দেওয়ার ভাষা নাই।।।।আল্লাহ...   
3  এক্কেরে হতাশ অই গেলাম জামাত-শিবিরের নাম নাই হি...   
4                        তোমায় হৃদয়ে রাখবো  সারাক্ষণ   

                                      english_speech          dialect  
0                               I got a job in Dhaka       Mymensingh  
1                    Everyone started walking again.       Chittagong  
2  What can I say... There are no words to expres...           Sylhet  
3  I am very disappointed that the name of Jamaat...         Noakhali  
4                I will keep you in my heart forever  Standard Bangla  
DataFrame shape: (53830, 3)


In [12]:
import pandas as pd

def load_translation_dataset(csv_path, bangla_col):
    """
    Load a translation CSV and return a standardized DataFrame.

    Args:
        csv_path (str): Path to the CSV file.
        bangla_col (str): Name of the Bangla sentence column in the CSV.
        english_col (str): Name of the English sentence column in the CSV.

    Returns:
        pd.DataFrame: DataFrame with columns ['bangla_speech', 'english_speech'].
    """
    english_col = "english_speech"
    # Load CSV
    df = pd.read_csv(csv_path)

    # Strip whitespace from column names
    df.columns = df.columns.str.strip()

    # Keep only the relevant columns
    df = df[[bangla_col.strip(), english_col.strip()]]

    # Rename columns to standard names
    df = df.rename(columns={bangla_col.strip(): "bangla_speech",
                            english_col.strip(): "english_speech"})

    return df



In [13]:
# Paths to the validation CSVs
barishal_csv = "/kaggle/input/vashantor-dialect-to-english/Vashantor /Validation/Barishal  Validation Translation.csv"
chittagong_csv = "/kaggle/input/vashantor-dialect-to-english/Vashantor /Validation/Chittagong Validation Translation.csv"
mymensingh_csv = "/kaggle/input/vashantor-dialect-to-english/Vashantor /Validation/Mymensingh Validation Translation.csv"
noakhali_csv = "/kaggle/input/vashantor-dialect-to-english/Vashantor /Validation/Noakhali Validation Translation.csv"
sylhet_csv = "/kaggle/input/vashantor-dialect-to-english/Vashantor /Validation/Sylhet Validation Translation.csv"

# Load datasets using your function
barishal_valid_df = load_translation_dataset(barishal_csv, "barishal_bangla_speech ")
chittagong_valid_df = load_translation_dataset(chittagong_csv, "chittagong_bangla_speech ")
mymensingh_valid_df = load_translation_dataset(mymensingh_csv, "mymensingh_bangla_speech ")
noakhali_valid_df = load_translation_dataset(noakhali_csv, "noakhali_bangla_speech ")
sylhet_valid_df = load_translation_dataset(sylhet_csv, "sylhet_bangla_speech ")
standard_valid_df = load_translation_dataset(sylhet_csv, "bangla_speech")

In [14]:
barishal_test_csv = "/kaggle/input/vashantor-dialect-to-english/Vashantor /Test/Barishal Test Translation.csv"
chittagong_test_csv = "/kaggle/input/vashantor-dialect-to-english/Vashantor /Test/Chittagong Test Translation.csv"
mymensingh_test_csv = "/kaggle/input/vashantor-dialect-to-english/Vashantor /Test/Mymensingh Test Translation.csv"
noakhali_test_csv = "/kaggle/input/vashantor-dialect-to-english/Vashantor /Test/Noakhali Test Translation.csv"
sylhet_test_csv = "/kaggle/input/vashantor-dialect-to-english/Vashantor /Test/Sylhet Test Translation.csv"
standard_test_csv = "/kaggle/input/vashantor-dialect-to-english/Vashantor /Test/Sylhet Test Translation.csv"

# Load test datasets
barishal_test_df = load_translation_dataset(barishal_test_csv, "barishal_bangla_speech ")
chittagong_test_df = load_translation_dataset(chittagong_test_csv, "chittagong_bangla_speech ")
mymensingh_test_df = load_translation_dataset(mymensingh_test_csv, "mymensingh_bangla_speech ")
noakhali_test_df = load_translation_dataset(noakhali_test_csv, "noakhali_bangla_speech ")
sylhet_test_df = load_translation_dataset(sylhet_test_csv, "sylhet_bangla_speech ")
standard_test_df = load_translation_dataset(standard_test_csv, "bangla_speech")

In [15]:
# barishal_train_csv = "/kaggle/input/vashantor-dialect-to-english/Vashantor /Train/Barishal Train Translation.csv"
# train_df = load_translation_dataset(barishal_train_csv, "barishal_bangla_speech ")

train_dataset = tokenize_and_create_dataset(tokenizer, train_df)


=== Checking types of source and target texts ===
Example 1: Source type=<class 'str'>, Target type=<class 'str'>
Example 2: Source type=<class 'str'>, Target type=<class 'str'>
Example 3: Source type=<class 'str'>, Target type=<class 'str'>
Example 4: Source type=<class 'str'>, Target type=<class 'str'>
Example 5: Source type=<class 'str'>, Target type=<class 'str'>

=== First 3 tokenized examples ===

Example 1:
Input : ঢাকায় আমার চাকরি হইসে
Label : I got a job in Dhaka

Example 2:
Input : আবার চল্যিত শুরু গরিল বেজ্ঞুনে
Label : Everyone started walking again.

Example 3:
Input : কিতা কইতাম ... বাহবা দেওয়ার ভাষা নাই।।।।আল্লাহ তুমরার বালা করুক্কা
Label : What can I say... There are no words to express how wonderful it is. May Allah bless you all.


In [17]:
barishal_valid_dataset = tokenize_and_create_dataset(tokenizer,barishal_valid_df)
chittagong_valid_dataset = tokenize_and_create_dataset(tokenizer, chittagong_valid_df)
mymensingh_valid_dataset = tokenize_and_create_dataset(tokenizer, mymensingh_valid_df)
noakhali_valid_dataset = tokenize_and_create_dataset(tokenizer, noakhali_valid_df)
sylhet_valid_dataset = tokenize_and_create_dataset(tokenizer, sylhet_valid_df)
standard_valid_dataset = tokenize_and_create_dataset(tokenizer, standard_valid_df)


=== Checking types of source and target texts ===
Example 1: Source type=<class 'str'>, Target type=<class 'str'>
Example 2: Source type=<class 'str'>, Target type=<class 'str'>
Example 3: Source type=<class 'str'>, Target type=<class 'str'>
Example 4: Source type=<class 'str'>, Target type=<class 'str'>
Example 5: Source type=<class 'str'>, Target type=<class 'str'>

=== First 3 tokenized examples ===

Example 1:
Input : বাংলাদেশে ৬৪ ডা জেলা
Label : 64 districts in Bangladesh

Example 2:
Input : মোরা হগলডি গ্যাছেকাইল বায়রায় গেল্লাহম
Label : We all went out yesterday

Example 3:
Input : তোমার কথা কওয়ার বাইল বেমালা সুন্দর
Label : Your way of speaking is very nice

=== Checking types of source and target texts ===
Example 1: Source type=<class 'str'>, Target type=<class 'str'>
Example 2: Source type=<class 'str'>, Target type=<class 'str'>
Example 3: Source type=<class 'str'>, Target type=<class 'str'>
Example 4: Source type=<class 'str'>, Target type=<class 'str'>
Example 5: Source 

In [16]:
barishal_test_dataset = tokenize_and_create_dataset(tokenizer,barishal_test_df)
chittagong_test_dataset = tokenize_and_create_dataset(tokenizer,chittagong_test_df)
mymensingh_test_dataset = tokenize_and_create_dataset(tokenizer,mymensingh_test_df)
noakhali_test_dataset = tokenize_and_create_dataset(tokenizer,noakhali_test_df)
sylhet_test_dataset = tokenize_and_create_dataset(tokenizer,sylhet_test_df)
standard_test_dataset = tokenize_and_create_dataset(tokenizer,standard_test_df)


=== Checking types of source and target texts ===
Example 1: Source type=<class 'str'>, Target type=<class 'str'>
Example 2: Source type=<class 'str'>, Target type=<class 'str'>
Example 3: Source type=<class 'str'>, Target type=<class 'str'>
Example 4: Source type=<class 'str'>, Target type=<class 'str'>
Example 5: Source type=<class 'str'>, Target type=<class 'str'>

=== First 3 tokenized examples ===

Example 1:
Input : তোমার আব্বায় ক্যামন আছে?
Label : How is your father?

Example 2:
Input : মোর বড় বুইনের আইজগো মন ভালো নাই
Label : My elder sister is not feeling well today

Example 3:
Input : তুমি কি মোরে এই কামডা কইররা দেতে পারবা?
Label : Can you do this for me?

=== Checking types of source and target texts ===
Example 1: Source type=<class 'str'>, Target type=<class 'str'>
Example 2: Source type=<class 'str'>, Target type=<class 'str'>
Example 3: Source type=<class 'str'>, Target type=<class 'str'>
Example 4: Source type=<class 'str'>, Target type=<class 'str'>
Example 5: Source

In [ ]:
import shutil
import os

# Delete the cached evaluate metrics folder
cache_dir = os.path.expanduser("~/.cache/huggingface/modules/evaluate_modules")
shutil.rmtree(cache_dir, ignore_errors=True)
print("✅ Hugging Face evaluate cache cleared.")

In [18]:
import sacrebleu

def normalize_texts(texts):
    normalized = []
    for t in texts:
        t = t.strip()
        t = " ".join(t.split())  # normalize whitespace
        normalized.append(t)
    return normalized

def decode_strip_pad(tokenizer, sequences):
    """
    Decode token IDs into strings after removing padding tokens or -100 labels.
    sequences: list of list of token IDs
    """
    decoded_texts = []
    for seq in sequences:
        seq = [tok for tok in seq if tok not in [tokenizer.pad_token_id, -100]]
        decoded_texts.append(tokenizer.decode(seq, skip_special_tokens=True, clean_up_tokenization_spaces=True))
    return decoded_texts

def compute_metrics(eval_pred):
    preds, labels = eval_pred

    # preds are already generated token IDs
    decoded_preds = decode_strip_pad(tokenizer, preds)
    decoded_labels = decode_strip_pad(tokenizer, labels)
    
    decoded_preds = normalize_texts(decoded_preds)
    decoded_labels = normalize_texts(decoded_labels)

    # sacrebleu expects list of references per prediction
    references = [decoded_labels]

    bleu_score = sacrebleu.corpus_bleu(decoded_preds, references, tokenize="13a").score
    return {"bleu": bleu_score}


In [ ]:
# def compute_bleu(preds, labels, tokenizer):
#     """
#     preds: numpy array of predicted token IDs
#     labels: numpy array of label token IDs
#     tokenizer: your Hugging Face tokenizer
#     """
#     # decode predictions
#     decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    
#     # decode labels
#     decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
#     # sacrebleu expects a list of references for each sentence
#     decoded_labels = [[label] for label in decoded_labels]
    
#     result = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
#     return result["score"]


In [ ]:
# def decode_strip_pad(tokenizer, sequences):
#     """
#     Decode token IDs into strings after removing padding tokens.
#     sequences: list of list of token IDs
#     """
#     decoded_texts = []
#     for seq in sequences:
#         # Remove padding tokens before decoding
#         seq = [tok for tok in seq if tok != tokenizer.pad_token_id]
#         decoded_texts.append(tokenizer.decode(seq, skip_special_tokens=True))
#     return decoded_texts

In [ ]:
# def compute_metrics(eval_pred):
#     preds, labels = eval_pred
#     decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
#     decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
#     decoded_labels = [[label] for label in decoded_labels]  # sacrebleu format
#     result = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
# #     return {"bleu": result["score"]}

# def compute_metrics(eval_pred):
#     preds, labels = eval_pred

#     # Remove padding before decoding
#     decoded_preds = decode_strip_pad(tokenizer, preds)
#     decoded_labels = decode_strip_pad(tokenizer, labels)

#     # sacrebleu expects a list of references for each prediction
#     decoded_labels = [[label] for label in decoded_labels]

#     result = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
#     return {"bleu": result["score"]}

In [19]:
val_datasets = {
    "Barishal": barishal_valid_dataset,
    "Chittagong": chittagong_valid_dataset,
    "Mymensingh": mymensingh_valid_dataset,
    "Noakhali": noakhali_valid_dataset,
    "Sylhet": sylhet_valid_dataset,
    "Standard": standard_valid_dataset
}

from datasets import concatenate_datasets

eval_dataset = concatenate_datasets([
    barishal_valid_dataset,
    chittagong_valid_dataset,
    mymensingh_valid_dataset,
    noakhali_valid_dataset,
    sylhet_valid_dataset,
    standard_valid_dataset
])

In [20]:

from transformers import TrainerCallback, TrainerState, TrainerControl

class MultiValidationBLEUCallback(TrainerCallback):
    """
    Evaluates the model on multiple validation datasets using BLEU at the end of each epoch.
    Also prints the first 5 sentences, their references, and predictions.
    """
    def __init__(self, val_datasets, trainer, compute_metrics_fn, tokenizer):
        self.val_datasets = val_datasets
        self.trainer = trainer
        self.compute_metrics_fn = compute_metrics_fn
        self.tokenizer = tokenizer

    def on_epoch_end(self, args, state: TrainerState, control: TrainerControl, **kwargs):
        print(f"\n=== BLEU Evaluation at epoch {state.epoch:.2f} ===")
        for name, dataset in self.val_datasets.items():
            # Make predictions using the trainer
            outputs = self.trainer.predict(dataset)
            eval_pred = (outputs.predictions, outputs.label_ids)
            
            # Compute BLEU
            result = self.compute_metrics_fn(eval_pred)
            print(f"{name} BLEU: {result['bleu']:.2f}\n")

            # Decode first 5 predictions and labels
            decoded_preds = decode_strip_pad(self.tokenizer, outputs.predictions[:5])
            decoded_labels = decode_strip_pad(self.tokenizer, outputs.label_ids[:5])
            # Get source sentences if available in dataset
            sources = dataset[:5]["input_ids"] if "input_ids" in dataset.column_names else ["<source not available>"] * 5
            decoded_sources = decode_strip_pad(self.tokenizer, sources)

            # Print first 5 examples
            print(f"--- First 5 examples for {name} ---")
            for i in range(len(decoded_preds)):
                print(f"Source    : {decoded_sources[i]}")
                print(f"Reference : {decoded_labels[i]}")
                print(f"Prediction: {decoded_preds[i]}")
                print("-" * 50)
                
        return control




In [22]:
unfreeze_schedule = {
    1: 2,     # unfreeze top 2 layers at epoch 1
    2: 4,     # unfreeze top 4 layers at epoch 2
    3: 'all'  # unfreeze all layers at epoch 3
}

gradual_unfreeze_cb = GradualUnfreezingCallback(model=model, unfreeze_schedule=unfreeze_schedule)


In [23]:
model_args = Seq2SeqTrainingArguments(
    output_dir="./output_dir",
    per_device_train_batch_size=8,       # Increased from 1
    gradient_accumulation_steps=4,       # Effective batch size = 64 (8x4x2)
    per_device_eval_batch_size=8,
    learning_rate=5e-5,                  # Slightly higher for faster convergence
    warmup_ratio=0.1,
    num_train_epochs=4,                  
    weight_decay=0.01,
    lr_scheduler_type="linear",
    eval_strategy="epoch",   # Evaluate at the end of each epoch
    logging_steps=200,
    save_strategy="epoch",
    metric_for_best_model="eval_loss",  # ✅ cross-entropy
    greater_is_better=False, 
    predict_with_generate=True,
    save_only_model=True,  
    fp16=True,                           # CRITICAL: Use Mixed Precision (if using NVIDIA GPU)
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    tokenizer=tokenizer,
    args=model_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics, 
    callbacks=[
        # EarlyStoppingCallback(
        #     early_stopping_patience=2,
        #     early_stopping_threshold=0.0
        # ),
        gradual_unfreeze_cb
        
    ],
)

# Usage
multi_val_bleu_cb = MultiValidationBLEUCallback(
    val_datasets=val_datasets,
    trainer=trainer,
    compute_metrics_fn=compute_metrics,  # your function from before
    tokenizer = tokenizer
)
# And add it to trainer callbacks
trainer.add_callback(multi_val_bleu_cb)
trainer.train()

/tmp/ipykernel_55/3429284229.py:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Bleu
1,1.653500,1.124647,28.624211
2,1.442800,1.047688,29.864677
3,1.303100,1.011307,31.156939
4,1.242400,1.005999,31.368502



=== BLEU Evaluation at epoch 1.00 ===
Barishal BLEU: 27.37

--- First 5 examples for Barishal ---
Source    : বাংলাদেশে ৬৪ ডা জেলা
Reference : 64 districts in Bangladesh
Prediction: District 64 in Bangladesh
--------------------------------------------------
Source    : মোরা হগলডি গ্যাছেকাইল বায়রায় গেল্লাহম
Reference : We all went out yesterday
Prediction: I will go to the village of Geshkel in the morning.
--------------------------------------------------
Source    : তোমার কথা কওয়ার বাইল বেমালা সুন্দর
Reference : Your way of speaking is very nice
Prediction: You are very beautiful to speak to.
--------------------------------------------------
Source    : বরিশালের মানু ক্যামন হয়?
Reference : How are the people of Barisal?
Prediction: How is the rainbow man?
--------------------------------------------------
Source    : খুলনা জেলা কি বেমালা সুন্দর?
Reference : Khulna district is very beautiful?
Prediction: Is Khulna district beautiful?
--------------------------------------------

/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(



=== BLEU Evaluation at epoch 2.00 ===
Barishal BLEU: 30.82

--- First 5 examples for Barishal ---
Source    : বাংলাদেশে ৬৪ ডা জেলা
Reference : 64 districts in Bangladesh
Prediction: Bangladesh has 64 districts
--------------------------------------------------
Source    : মোরা হগলডি গ্যাছেকাইল বায়রায় গেল্লাহম
Reference : We all went out yesterday
Prediction: We will go to the village for a walk.
--------------------------------------------------
Source    : তোমার কথা কওয়ার বাইল বেমালা সুন্দর
Reference : Your way of speaking is very nice
Prediction: You are very beautiful to speak of
--------------------------------------------------
Source    : বরিশালের মানু ক্যামন হয়?
Reference : How are the people of Barisal?
Prediction: How is the rainbow man?
--------------------------------------------------
Source    : খুলনা জেলা কি বেমালা সুন্দর?
Reference : Khulna district is very beautiful?
Prediction: Is Khulna district beautiful?
--------------------------------------------------
Chitta

TrainOutput(global_step=3368, training_loss=1.4988180871813994, metrics={'train_runtime': 19411.9163, 'train_samples_per_second': 11.092, 'train_steps_per_second': 0.174, 'total_flos': 5.832762036584448e+16, 'train_loss': 1.4988180871813994, 'epoch': 4.0})

In [24]:
# Suppose your trainer is called `trainer`
trainer.save_model("./finetuned_nllb600m")
tokenizer.save_pretrained("./finetuned_nllb600m")


('./finetuned_nllb600m/tokenizer_config.json',
 './finetuned_nllb600m/special_tokens_map.json',
 './finetuned_nllb600m/sentencepiece.bpe.model',
 './finetuned_nllb600m/added_tokens.json',
 './finetuned_nllb600m/tokenizer.json')

In [27]:
!zip -r my_folder.zip /kaggle/working/output_dir

  adding: kaggle/working/output_dir/ (stored 0%)
  adding: kaggle/working/output_dir/checkpoint-441/ (stored 0%)
  adding: kaggle/working/output_dir/checkpoint-441/trainer_state.json (deflated 63%)
  adding: kaggle/working/output_dir/checkpoint-441/training_args.bin (deflated 53%)
  adding: kaggle/working/output_dir/checkpoint-441/model.safetensors

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


^C



zip error: Interrupted (aborting)


In [26]:
!mv -r ./output_dir /kaggle/output/

mv: invalid option -- 'r'
Try 'mv --help' for more information.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [28]:
!zip -r /kaggle/working/model_checkpoints.zip /kaggle/working/output_dir

  adding: kaggle/working/output_dir/ (stored 0%)
  adding: kaggle/working/output_dir/checkpoint-441/ (stored 0%)
  adding: kaggle/working/output_dir/checkpoint-441/trainer_state.json (deflated 63%)
  adding: kaggle/working/output_dir/checkpoint-441/training_args.bin (deflated 53%)
  adding: kaggle/working/output_dir/checkpoint-441/model.safetensors

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 30%)
  adding: kaggle/working/output_dir/checkpoint-441/generation_config.json (deflated 32%)
  adding: kaggle/working/output_dir/checkpoint-441/config.json (deflated 57%)
  adding: kaggle/working/output_dir/checkpoint-441/tokenizer.json (deflated 82%)
  adding: kaggle/working/output_dir/checkpoint-441/sentencepiece.bpe.model (deflated 51%)
  adding: kaggle/working/output_dir/checkpoint-441/scheduler.pt (deflated 61%)
  adding: kaggle/working/output_dir/checkpoint-441/optimizer.pt
zip I/O error: No space left on device
zip error: Output file write failure (write error on zip file)


In [3]:
from IPython.display import display, HTML

display(HTML('<a href="/kaggle/working/model_checkpoints.zip" target="_blank">Download Model Checkpoints</a>'))


In [4]:
!cp /kaggle/working/model_checkpoints.zip /kaggle/temp/

cp: cannot create regular file '/kaggle/temp/': Not a directory


In [ ]:
torch.cuda.empty_cache()

In [ ]:
import numpy as np

# 1. Preprocessing with -100 for padding labels
def preprocess_labels(labels_ids):
    # Replace tokenizer padding token id with -100
    return [[(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels_ids]

In [ ]:
model_args = Seq2SeqTrainingArguments(
    output_dir="./eval_results",
    per_device_eval_batch_size=16,      # Much faster than 1
    predict_with_generate=True,        # Necessary for real translation eval
    fp16=True,                         # Use if you have an NVIDIA GPU
    report_to="none"
)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model, 
    args=model_args, 
    tokenizer=tokenizer
)

In [1]:
# Install requirements
!pip install transformers sacrebleu tqdm pandas torch

# Optional: Install the better normalizer
!pip install git+https://github.com/csebuetnlp/normalizer


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.6 MB/s eta 0:00:00
  Cloning https://github.com/csebuetnlp/normalizer to /tmp/pip-req-build-noxuwpop
  Running command git clone --filter=blob:none --quiet https://github.com/csebuetnlp/normalizer /tmp/pip-req-build-noxuwpop
  Resolved https://github.com/csebuetnlp/normalizer to commit d405944dde5ceeacb7c2fd3245ae2a9dea5f35c9
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 4.5 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for normalizer: filename=normalizer-0.0.1-py3-none-any.whl size=6860 sha256=d753cec76562d2f866c5f63085e51ecc2bc9a6f02a896f1a94a0d8bf90b27589
  Stored in directory: /tmp/pip-ephem-wheel-cache-2raxqc3g/wheels/f9/d8/55/a13fa77440d3e80bf10ff80176ba67c7a0543a67827ef0b8eb
  Created wheel for emoji: filename=emo

In [2]:
!pip install git+https://github.com/csebuetnlp/normalizer

  Cloning https://github.com/csebuetnlp/normalizer to /tmp/pip-req-build-_19q3fb3
  Running command git clone --filter=blob:none --quiet https://github.com/csebuetnlp/normalizer /tmp/pip-req-build-_19q3fb3
  Resolved https://github.com/csebuetnlp/normalizer to commit d405944dde5ceeacb7c2fd3245ae2a9dea5f35c9
  Preparing metadata (setup.py) ... done


In [3]:
import torch
import pandas as pd
import sacrebleu
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm
import numpy as np
import time
import os

# ------------------------------------------------------------
# 1. SETUP
# ------------------------------------------------------------
print("🔧 Setting up...")
device = torch.device("cpu")
torch.set_num_threads(os.cpu_count())
print(f"   Using {torch.get_num_threads()} CPU threads")

# ------------------------------------------------------------
# 2. LOAD MODEL
# ------------------------------------------------------------
print("\n📥 Loading model...")
model_name = "csebuetnlp/banglat5_nmt_bn_en"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model = model.to(device)
model.eval()
print("✅ Model loaded successfully")

# ------------------------------------------------------------
# 3. DATA LOADING FUNCTION
# ------------------------------------------------------------
def load_dialect_dataset(csv_path, dialect_name):
    """Load dataset for a specific dialect"""
    df = pd.read_csv(csv_path)
    
    # Based on your debug output, columns have trailing spaces
    df.columns = [col.strip() for col in df.columns]
    
    # Different dialects have different Bangla column names
    if dialect_name == "Barishal":
        bangla_col = "barishal_bangla_speech"
    elif dialect_name == "Chittagong":
        bangla_col = "chittagong_bangla_speech"
    elif dialect_name == "Mymensingh":
        bangla_col = "mymensingh_bangla_speech"
    elif dialect_name == "Noakhali":
        bangla_col = "noakhali_bangla_speech"
    elif dialect_name == "Sylhet":
        bangla_col = "sylhet_bangla_speech"
    else:
        bangla_col = "bangla_speech"  # fallback
    
    # Check if column exists
    if bangla_col not in df.columns:
        print(f"   Warning: {bangla_col} not found. Available columns: {list(df.columns)}")
        # Try to find any Bangla column
        bangla_cols = [col for col in df.columns if 'bangla' in col.lower()]
        if bangla_cols:
            bangla_col = bangla_cols[0]
            print(f"   Using alternative: {bangla_col}")
    
    english_col = "english_speech"
    
    # Extract data
    bangla_texts = df[bangla_col].astype(str).tolist()
    english_texts = df[english_col].astype(str).tolist()
    
    return bangla_texts, english_texts

# ------------------------------------------------------------
# 4. NORMALIZER
# ------------------------------------------------------------
def normalize_bangla(text):
    """Simple normalization (you can add the csebuetnlp normalizer here)"""
    text = str(text).strip()
    text = ' '.join(text.split())  # Remove extra whitespace
    return text

# Try to use the advanced normalizer
try:
    from normalizer import normalize as bangla_normalize
    print("✅ Using advanced Bangla normalizer")
    
    def normalize_bangla(text):
        text = str(text).strip()
        if text:
            return bangla_normalize(text)
        return ""
except ImportError:
    print("⚠️  Using simple normalizer")

# ------------------------------------------------------------
# 5. TRANSLATION FUNCTION
# ------------------------------------------------------------
def translate_batch(texts, model, tokenizer, device, batch_size=4):
    """Translate a batch of texts"""
    translations = []
    
    for i in tqdm(range(0, len(texts), batch_size), 
                  desc="Translating", 
                  unit="batch",
                  leave=False):
        batch = texts[i:i+batch_size]
        
        # Normalize
        normalized_batch = [normalize_bangla(text) for text in batch]
        
        # Tokenize
        inputs = tokenizer(
            normalized_batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128
        ).to(device)
        
        # Generate
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=128,
                num_beams=5,
                # early_stopping=True,
                # no_repeat_ngram_size=2
            )
        
        # Decode
        batch_translations = tokenizer.batch_decode(
            outputs, 
            skip_special_tokens=True
        )
        translations.extend(batch_translations)
    
    return translations

# ------------------------------------------------------------
# 6. VALIDATION DATASETS
# ------------------------------------------------------------
val_paths = {
    "Barishal": "/kaggle/input/vashantor-dialect-to-english/Vashantor /Validation/Barishal  Validation Translation.csv",
    "Chittagong": "/kaggle/input/vashantor-dialect-to-english/Vashantor /Validation/Chittagong Validation Translation.csv", 
    "Mymensingh": "/kaggle/input/vashantor-dialect-to-english/Vashantor /Validation/Mymensingh Validation Translation.csv",
    "Noakhali": "/kaggle/input/vashantor-dialect-to-english/Vashantor /Validation/Noakhali Validation Translation.csv",
    "Sylhet": "/kaggle/input/vashantor-dialect-to-english/Vashantor /Validation/Sylhet Validation Translation.csv"
}

# ------------------------------------------------------------
# 7. RUN INFERENCE ON ALL DATASETS
# ------------------------------------------------------------
print("\n" + "="*60)
print("🚀 STARTING FULL INFERENCE")
print("="*60)

results = {}
total_start_time = time.time()

for dialect_name, csv_path in val_paths.items():
    print(f"\n📊 Processing {dialect_name}...")
    
    # Load data
    bangla_texts, english_refs = load_dialect_dataset(csv_path, dialect_name)
    
    print(f"   Loaded {len(bangla_texts)} sentences")
    print(f"   Sample: '{bangla_texts[0][:50]}...'")
    
    # Translate
    start_time = time.time()
    translations = translate_batch(
        bangla_texts, 
        model, 
        tokenizer, 
        device,
        batch_size=4  # Good for CPU
    )
    translation_time = time.time() - start_time
    
    # Compute BLEU
    refs = [[ref] for ref in english_refs]
    bleu_score = sacrebleu.corpus_bleu(translations, refs).score
    
    # Store results
    results[dialect_name] = {
        "bleu": bleu_score,
        "translations": translations,
        "references": english_refs,
        "sources": bangla_texts,
        "time_seconds": translation_time
    }
    
    print(f"   ✅ BLEU Score: {bleu_score:.2f}")
    print(f"   ⏱️  Time: {translation_time:.1f}s ({translation_time/len(bangla_texts):.2f}s per sentence)")

# ------------------------------------------------------------
# 8. SAVE RESULTS
# ------------------------------------------------------------
print("\n" + "="*60)
print("💾 SAVING RESULTS")
print("="*60)

os.makedirs("./inference_results", exist_ok=True)

# Save detailed translations for each dialect
for dialect_name, result in results.items():
    df_output = pd.DataFrame({
        "bangla_source": result["sources"],
        "english_reference": result["references"],
        "english_prediction": result["translations"]
    })
    
    output_path = f"./inference_results/{dialect_name}_translations.csv"
    df_output.to_csv(output_path, index=False, encoding='utf-8')
    print(f"   ✓ Saved {dialect_name}: {output_path}")

# Save summary
summary_data = []
for dialect_name, result in results.items():
    summary_data.append({
        "Dialect": dialect_name,
        "BLEU_Score": result["bleu"],
        "Samples": len(result["sources"]),
        "Time_Seconds": result["time_seconds"],
        "Time_Per_Sentence": result["time_seconds"] / len(result["sources"])
    })

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv("./inference_results/summary.csv", index=False)
print(f"   ✓ Saved summary: ./inference_results/summary.csv")

# ------------------------------------------------------------
# 9. PRINT FINAL RESULTS
# ------------------------------------------------------------
print("\n" + "="*60)
print("📈 FINAL RESULTS")
print("="*60)

print("\nDialect            BLEU   Samples   Time")
print("-" * 45)

for dialect_name, result in results.items():
    print(f"{dialect_name:15} {result['bleu']:6.2f} {len(result['sources']):8} {result['time_seconds']:6.1f}s")

total_time = time.time() - total_start_time
total_sentences = sum(len(r["sources"]) for r in results.values())
avg_bleu = np.mean([r["bleu"] for r in results.values()])

print("\n" + "-" * 45)
print(f"TOTAL              {avg_bleu:6.2f} {total_sentences:8} {total_time:6.1f}s")
print(f"Average time per sentence: {total_time/total_sentences:.2f}s")

# ------------------------------------------------------------
# 10. SHOW SAMPLES
# ------------------------------------------------------------
print("\n" + "="*60)
print("🔍 SAMPLE TRANSLATIONS")
print("="*60)

for dialect_name, result in results.items():
    print(f"\n{dialect_name} (BLEU: {result['bleu']:.2f}):")
    for i in range(min(3, len(result["sources"]))):
        print(f"  [{i+1}] Source: {result['sources'][i][:60]}...")
        print(f"      Reference: {result['references'][i]}")
        print(f"      Prediction: {result['translations'][i]}")
        print()

print("\n✅ Inference complete! Results saved to ./inference_results/")

🔧 Setting up...
   Using 4 CPU threads

📥 Loading model...


tokenizer_config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/766 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

   ✅ BLEU Score: 100.00
   ⏱️  Time: 145.9s (0.58s per sentence)

📊 Processing Chittagong...
   Loaded 250 sentences
   Sample: 'বাংলাদেশত ৬৪ ইয়ান জেলা...'


   ✅ BLEU Score: 23.64
   ⏱️  Time: 146.7s (0.59s per sentence)

📊 Processing Mymensingh...
   Loaded 250 sentences
   Sample: 'বাংলাদেশে ৬৪ টা জেলা...'


   ✅ BLEU Score: 100.00
   ⏱️  Time: 109.4s (0.44s per sentence)

📊 Processing Noakhali...
   Loaded 250 sentences
   Sample: 'বাংলাদেশে ৬৪ গা জেলা...'


   ✅ BLEU Score: 42.73
   ⏱️  Time: 120.7s (0.48s per sentence)

📊 Processing Sylhet...
   Loaded 250 sentences
   Sample: 'বাংলাদেশো ৬৪ টা জেলা...'


   ✅ BLEU Score: 37.99
   ⏱️  Time: 133.7s (0.53s per sentence)

💾 SAVING RESULTS
   ✓ Saved Barishal: ./inference_results/Barishal_translations.csv
   ✓ Saved Chittagong: ./inference_results/Chittagong_translations.csv
   ✓ Saved Mymensingh: ./inference_results/Mymensingh_translations.csv
   ✓ Saved Noakhali: ./inference_results/Noakhali_translations.csv
   ✓ Saved Sylhet: ./inference_results/Sylhet_translations.csv
   ✓ Saved summary: ./inference_results/summary.csv

📈 FINAL RESULTS

Dialect            BLEU   Samples   Time
---------------------------------------------
Barishal        100.00      250  145.9s
Chittagong       23.64      250  146.7s
Mymensingh      100.00      250  109.4s
Noakhali         42.73      250  120.7s
Sylhet           37.99      250  133.7s

---------------------------------------------
TOTAL               60.87     1250  656.5s
Average time per sentence: 0.53s

🔍 SAMPLE TRANSLATIONS

Barishal (BLEU: 100.00):
  [1] Source: বাংলাদেশে ৬৪ ডা জেলা...
      Refere

In [15]:
import torch
import pandas as pd
import sacrebleu
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm
import numpy as np
import time
import os

# ✅ BUET NORMALIZER
from normalizer import normalize

# ================ 1. SETUP ================
print("🔧 Setting up for inference...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(os.cpu_count())
print(f"   Device: {device}, Threads: {torch.get_num_threads()}")

# ================ 2. LOAD MODEL ================
print("\n📥 Loading model 'csebuetnlp/banglat5_nmt_bn_en'...")
model_name = "csebuetnlp/banglat5_nmt_bn_en"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
model.eval()
print("✅ Model loaded.")

# ================ 3. DATA LOADER WITH VALIDATION ================
def load_and_validate_dataset(csv_path, dialect_name):
    print(f"   Reading: {os.path.basename(csv_path)}")
    df = pd.read_csv(csv_path)

    df.columns = [col.strip() for col in df.columns]
    print(f"   Found columns: {list(df.columns)}")

    dialect_col_candidates = [
        f"{dialect_name.lower()}bangla_speech",
        "bangla_speech"
    ]

    bangla_col = None
    for col in dialect_col_candidates:
        if col in df.columns:
            bangla_col = col
            break

    if bangla_col is None:
        raise ValueError("❌ Bangla source column not found.")

    english_col = "english_speech"
    if english_col not in df.columns:
        raise ValueError("❌ English reference column not found.")

    print(f"   Selected SOURCE column: '{bangla_col}'")
    print(f"   Selected REFERENCE column: '{english_col}'")

    bangla_texts = df[bangla_col].astype(str).tolist()
    english_texts = df[english_col].astype(str).tolist()

    print(f"   Sample -> Source: '{bangla_texts[0][:30]}...'")
    print(f"             Reference: '{english_texts[0][:30]}...'")

    return bangla_texts, english_texts

# ================ 4. TRANSLATION FUNCTION (WITH NORMALIZATION) ================
def translate_with_validation(bangla_texts, model, tokenizer, device, batch_size=4):
    translations = []

    for i in tqdm(range(0, len(bangla_texts), batch_size),
                  desc="   Translating",
                  unit="batch",
                  leave=False):

        # 🔹 NORMALIZATION HAPPENS HERE
        batch = bangla_texts[i:i + batch_size]
        batch = [normalize(str(x).strip()) for x in batch]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=128,
                num_beams=5
            )

        decoded = tokenizer.batch_decode(
            outputs,
            skip_special_tokens=True
        )

        for j, t in enumerate(decoded):
            if not t or t.isspace():
                print(f"   ⚠️ Empty output for: {batch[j][:30]}...")
                decoded[j] = "[EMPTY_OUTPUT]"

        translations.extend(decoded)

    return translations

# ================ 5. VALIDATION FILES ================
val_paths = {
    "Barishal_": "/kaggle/input/vashantor-dialect-to-english/Vashantor /Validation/Barishal  Validation Translation.csv",
    "Chittagong_": "/kaggle/input/vashantor-dialect-to-english/Vashantor /Validation/Chittagong Validation Translation.csv",
    "Mymensingh_": "/kaggle/input/vashantor-dialect-to-english/Vashantor /Validation/Mymensingh Validation Translation.csv",
    "Noakhali_": "/kaggle/input/vashantor-dialect-to-english/Vashantor /Validation/Noakhali Validation Translation.csv",
    "Sylhet_": "/kaggle/input/vashantor-dialect-to-english/Vashantor /Validation/Sylhet Validation Translation.csv",
    "": "/kaggle/input/vashantor-dialect-to-english/Vashantor /Validation/Barishal  Validation Translation.csv",
}

# ================ 6. MAIN LOOP ================
print("\n" + "=" * 60)
print("🚀 STARTING INFERENCE")
print("=" * 60)

results = {}

for dialect, path in val_paths.items():
    print(f"\n📊 [{dialect.upper()}]")

    source_texts, reference_texts = load_and_validate_dataset(path, dialect)

    start = time.time()
    predicted_texts = translate_with_validation(
        source_texts, model, tokenizer, device, batch_size=4
    )
    elapsed = time.time() - start

    print("\n🧪 FIRST 100 TRANSLATION SAMPLES")
    print("-" * 60)
    limit = min(100, len(source_texts))
    for i in range(limit):
        print(f"\n[{i+1}]")
        print(f"Source ({dialect}): {source_texts[i]}")
        print(f"Prediction       : {predicted_texts[i]}")
        print(f"Reference        : {reference_texts[i]}")
    print("-" * 60)

    bleu_score = sacrebleu.corpus_bleu(
        predicted_texts,
        [reference_texts]
    ).score

    exact_matches = sum(
        p == r for p, r in zip(predicted_texts, reference_texts)
    )

    print(f"   Exact matches: {exact_matches}/{len(reference_texts)}")
    print(f"   ✅ BLEU: {bleu_score:.2f}, Time: {elapsed:.1f}s")

    results[dialect] = {
        "bleu": bleu_score,
        "time": elapsed
    }

# ================ 7. SUMMARY ================
print("\n" + "=" * 60)
print("📊 FINAL BLEU SCORES")
print("=" * 60)

for d, r in results.items():
    print(f"{d:12}: BLEU = {r['bleu']:.2f}")


🔧 Setting up for inference...
   Device: cpu, Threads: 4

📥 Loading model 'csebuetnlp/banglat5_nmt_bn_en'...
✅ Model loaded.

🚀 STARTING INFERENCE

📊 [BARISHAL_]
   Reading: Barishal  Validation Translation.csv
   Found columns: ['bangla_speech', 'banglish_speech', 'barishal_bangla_speech', 'barishal_banglish_speech', 'region_name', 'english_speech']
   Selected SOURCE column: 'barishal_bangla_speech'
   Selected REFERENCE column: 'english_speech'
   Sample -> Source: 'বাংলাদেশে ৬৪ ডা জেলা...'
             Reference: '64 districts in Bangladesh...'



🧪 FIRST 100 TRANSLATION SAMPLES
------------------------------------------------------------

[1]
Source (Barishal_): বাংলাদেশে ৬৪ ডা জেলা
Prediction       : 64 districts in Bangladesh
Reference        : 64 districts in Bangladesh

[2]
Source (Barishal_): মোরা হগলডি গ্যাছেকাইল বায়রায় গেল্লাহম
Prediction       : Mora Hogli Geykail Bayray Geyllum
Reference        : We all went out yesterday

[3]
Source (Barishal_): তোমার কথা কওয়ার বাইল বেমালা সুন্দর
Prediction       : Your voice is beautiful.
Reference        : Your way of speaking is very nice

[4]
Source (Barishal_): বরিশালের মানু ক্যামন হয়?
Prediction       : What is Manu from Barisal?
Reference        : How are the people of Barisal?

[5]
Source (Barishal_): খুলনা জেলা কি বেমালা সুন্দর?
Prediction       : Is Khulna District Bemala Beautiful?
Reference        : Khulna district is very beautiful?

[6]
Source (Barishal_): খুলনার মানু হেইরহম সুন্দর হয় না
Prediction       : Manu Heirham of Khulna is not beautiful
Reference        : Khuln


🧪 FIRST 100 TRANSLATION SAMPLES
------------------------------------------------------------

[1]
Source (Chittagong_): বাংলাদেশত ৬৪ ইয়ান জেলা
Prediction       : 64 Yan District in Bangladesh
Reference        : 64 districts in Bangladesh

[2]
Source (Chittagong_): আরা বেয়াক্কুন গতহালিয়া বাইরে গেইলাম
Prediction       : Ara Beyakkun Gayalam outside
Reference        : We all went out yesterday

[3]
Source (Chittagong_): তোইয়ার হতা বলার ধরণ বহুত সুন্দর
Prediction       : Toy's a nice way of saying a slaughter.
Reference        : Your way of speaking is very nice

[4]
Source (Chittagong_): বরিশালর মানুষ হইল্লে অয় দে?
Prediction       : If he's from Barisal, why don't he?
Reference        : How are the people of Barisal?

[5]
Source (Chittagong_):  খুলনা জেলা কি বহুত সুন্দর নাকি?
Prediction       : Is Khulna a great district?
Reference        : Khulna district is very beautiful?

[6]
Source (Chittagong_):  খুলনার মানুষ এইল্লে সুন্দর ন অয়
Prediction       : The people of Khulna are not beaut


🧪 FIRST 100 TRANSLATION SAMPLES
------------------------------------------------------------

[1]
Source (Mymensingh_): বাংলাদেশে ৬৪ টা জেলা
Prediction       : 64 districts in Bangladesh
Reference        : 64 districts in Bangladesh

[2]
Source (Mymensingh_): আমরা বেহেই কালকে বাইরে গেসিলাম 
Prediction       : We went out yesterday.
Reference        : We all went out yesterday

[3]
Source (Mymensingh_): তোমার কথা কওয়ার ধরন মেলা সুন্দর 
Prediction       : You have a nice way of speaking.
Reference        : Your way of speaking is very nice

[4]
Source (Mymensingh_): বরিশালের মানুষ কেমত হয়? 
Prediction       : What are the people of Barisal like?
Reference        : How are the people of Barisal?

[5]
Source (Mymensingh_): খুলনা জেলা কিতা মেলা সুন্দর? 
Prediction       : Khulna District Kita Mela is beautiful?
Reference        : Khulna district is very beautiful?

[6]
Source (Mymensingh_): খুলনার মানুষ তেমন সুন্দর হয় না 
Prediction       : The people of Khulna are not so beautiful
Referen


🧪 FIRST 100 TRANSLATION SAMPLES
------------------------------------------------------------

[1]
Source (Noakhali_): বাংলাদেশে ৬৪ গা জেলা
Prediction       : 64 Ga districts in Bangladesh
Reference        : 64 districts in Bangladesh

[2]
Source (Noakhali_): আন্ডা বেজ্ঞুন কাইল্লা বাইরে গেসিলাম
Prediction       : Anda Benjun Killa went out
Reference        : We all went out yesterday

[3]
Source (Noakhali_): তোওয়ার কথা কইবার ধরন অনেক সুন্দর
Prediction       : It's a nice way to talk to you.
Reference        : Your way of speaking is very nice

[4]
Source (Noakhali_): বরিশালের মানুষ কেইচ্চা অয়?
Prediction       : Who's the Barisal man?
Reference        : How are the people of Barisal?

[5]
Source (Noakhali_): খুলনা জেলা কি অনেক সুন্দর?
Prediction       : Is Khulna very beautiful?
Reference        : Khulna district is very beautiful?

[6]
Source (Noakhali_): খুলনার মানুষ হেইচ্চা সুন্দর অয় না
Prediction       : The people of Khulna aren't very nice.
Reference        : Khulna people are no


🧪 FIRST 100 TRANSLATION SAMPLES
------------------------------------------------------------

[1]
Source (Sylhet_): বাংলাদেশো ৬৪ টা জেলা
Prediction       : 64 districts of Bangladesh
Reference        : 64 districts in Bangladesh

[2]
Source (Sylhet_): আমরা হকল গতকাইল বাহিরো গিয়েছিলাম
Prediction       : We went out on the hawk.
Reference        : We all went out yesterday

[3]
Source (Sylhet_): তোমার মাত বলার ধরন বহুত সুন্দর 
Prediction       : You have a nice way of saying Mata.
Reference        : Your way of speaking is very nice

[4]
Source (Sylhet_): বরিশালর মানুষ কিলা অইন?
Prediction       : Is Barisal's people Killa Ain?
Reference        : How are the people of Barisal?

[5]
Source (Sylhet_): খুলনা জেলা কিতা বহুত সুন্দর?  
Prediction       : Is Khulna very beautiful?
Reference        : Khulna district is very beautiful?

[6]
Source (Sylhet_): খুলনার মানুষ তেমন সুন্দর অয় না
Prediction       : The people of Khulna aren't so beautiful.
Reference        : Khulna people are not that n


🧪 FIRST 100 TRANSLATION SAMPLES
------------------------------------------------------------

[1]
Source (): বাংলাদেশে ৬৪ টা জেলা
Prediction       : 64 districts in Bangladesh
Reference        : 64 districts in Bangladesh

[2]
Source (): আমরা সবাই গতকাল বাহিরে গিয়েছিলাম 
Prediction       : We all went out yesterday.
Reference        : We all went out yesterday

[3]
Source (): তোমার কথা বলার ধরন অনেক সুন্দর 
Prediction       : You have a nice way of speaking.
Reference        : Your way of speaking is very nice

[4]
Source (): বরিশালের মানুষ কেমন হয়? 
Prediction       : How are the people of Barisal?
Reference        : How are the people of Barisal?

[5]
Source (): খুলনা জেলা কি অনেক সুন্দর? 
Prediction       : Is Khulna very beautiful?
Reference        : Khulna district is very beautiful?

[6]
Source (): খুলনার মানুষ তেমন সুন্দর হয় না 
Prediction       : The people of Khulna are not so beautiful
Reference        : Khulna people are not that nice

[7]
Source (): কখনো ভেবে দেখেছ আমি না 